# ResNet34-UNet + H95 — Colab Training (A100 / L4 / V100)

**Target hardware:** NVIDIA A100 40 GB (Colab High-RAM). Falls back to L4 / V100 with reduced batch and FP16. T4 is rejected.

**Approach:** Clones this repo from GitHub into `/content/RESNET_h95`, runs the `src/` pipeline locally, builds a Colab-tuned config, and calls each stage in process. **No source edits** — only this notebook changes.

**Canonical sources:**
- **Code:** GitHub `https://github.com/chautuanminh/RESNET_h95` → cloned to `/content/RESNET_h95`
- **Dataset:** DocTamper LMDBs at `/content/drive/MyDrive/usth/Final_Project/dtset/archive`
- **Encoder cache / checkpoints / results:** Google Drive

**Outputs** stream to `/content/drive/Othercomputers/MSI/results/RESNET_h95/runs/<RUN_DIR>/` (visible on the MSI machine via Drive for Desktop).

**One switch — `RUN_MODE`** (set in section 1): `smoke` | `train` | `resume` | `post_train` | `full`.

## 1. Mount Drive, paths & RUN_MODE — EDIT HERE

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# ─── EDIT HERE — the only cell you normally touch ────────────────────────────
REPO_URL        = 'https://github.com/chautuanminh/RESNET_h95.git'
REPO_DIR        = Path('/content/RESNET_h95')                                  # local clone (code runs from here)
DATA_ROOT       = Path('/content/drive/MyDrive/usth/Final_Project/dtset/archive')
OUTPUT_ROOT     = Path('/content/drive/Othercomputers/MSI/results/RESNET_h95/runs')
RUN_DIR         = 'doctamper_resnet34_h95_35epochs_comparison'
TAMPER_META     = REPO_DIR / 'tampering_types'                                 # committed in the repo
TORCH_HOME      = Path('/content/drive/Othercomputers/MSI/results/RESNET_h95/cache/torch')

RUN_MODE        = 'full'    # smoke | train | resume | post_train | full
EPOCHS_OVERRIDE = None      # int → overrides training.epochs; None keeps the config value (35)
REINSTALL_TORCH = False     # keep Colab's prebuilt CUDA torch unless you explicitly need a reinstall
# ──────────────────────────────────────────────────────────────────────

assert RUN_MODE in {'smoke', 'train', 'resume', 'post_train', 'full'}, f'bad RUN_MODE: {RUN_MODE}'
assert DATA_ROOT.exists(), f'data root not found: {DATA_ROOT}'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
TORCH_HOME.mkdir(parents=True, exist_ok=True)
os.environ['TORCH_HOME'] = str(TORCH_HOME)   # cache ImageNet encoder weights on Drive (set before torch import)

print(f'OK  RUN_MODE     : {RUN_MODE}')
print(f'OK  data root    : {DATA_ROOT}')
print(f'OK  output root  : {OUTPUT_ROOT}')
print(f'OK  TORCH_HOME   : {TORCH_HOME}')
print(f'(repo will be cloned to {REPO_DIR} in section 4)')

## 2. GPU detection — warn & fall back, do not hard-assert

Decides `amp_dtype` and `batch_size_candidates` from the GPU Colab actually allocated. Picked up by section 5 when the config is built.

In [ ]:
import torch

assert torch.cuda.is_available(), 'CUDA is not available — switch to a GPU runtime'
gpu_name       = torch.cuda.get_device_name(0)
total_vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
bf16_supported = torch.cuda.is_bf16_supported()
print(f'GPU:  {gpu_name}')
print(f'VRAM: {total_vram_gb:.2f} GB')
print(f'BF16: {bf16_supported}')

if 'A100' in gpu_name:
    AMP_DTYPE        = 'bf16'
    BATCH_CANDIDATES = [56, 60, 64, 72, 80, 96]
    MEMORY_FRACTION  = 0.85
    print('✓ A100 detected — BF16, candidates [56..96], memory_fraction 0.85')
elif 'L4' in gpu_name:
    # L4 is Ada Lovelace (compute 8.9): BF16 native, 22.5 GB VRAM.
    # Empirically ResNet34-UNet @ 512² uses ~340 MB/sample in the autotune probe,
    # so batch=48 ≈ 16 GB, batch=64 ≈ 22 GB — push to the largest that fits at 0.85.
    AMP_DTYPE        = 'bf16' if bf16_supported else 'fp16'
    BATCH_CANDIDATES = [48, 50, 56, 64]
    MEMORY_FRACTION  = 0.85
    print(f'✓ L4 detected ({total_vram_gb:.1f} GB) — {AMP_DTYPE.upper()}, candidates [48..64], memory_fraction 0.85')
elif 'V100' in gpu_name:
    AMP_DTYPE        = 'fp16'
    MEMORY_FRACTION  = 0.80
    if total_vram_gb >= 24:
        BATCH_CANDIDATES = [16, 24, 32, 48, 64]
        print(f'⚠  V100-32GB detected — FP16, candidates [16..64]')
    else:
        BATCH_CANDIDATES = [8, 16, 24, 32]
        print(f'⚠  V100-16GB detected — FP16, candidates [8..32]')
elif 'T4' in gpu_name:
    raise RuntimeError(
        f'{gpu_name} only has ~16 GB — insufficient for ResNet34-UNet at 512x512. '
        'Disconnect and request a larger GPU.'
    )
else:
    AMP_DTYPE        = 'bf16' if bf16_supported else 'fp16'
    BATCH_CANDIDATES = [8, 16, 24, 32]
    MEMORY_FRACTION  = 0.80
    print(f'⚠  Unknown GPU {gpu_name} — defensive defaults: {AMP_DTYPE}, candidates [8..32]')

## 3. Run mode → stage gating

`RUN_MODE` (set in section 1) selects which stages run:

| RUN_MODE     | smoke | train | resume | eval / failure / tamper | checkpoint used |
|--------------|:-----:|:-----:|:------:|:-----------------------:|-----------------|
| `smoke`      |  ✓   |  –   |  –   |  –  | – |
| `train`      |  ✓   |  ✓ (fresh) |  –   |  –  | – |
| `resume`     |  –   |  ✓   |  ✓   |  –  | `last_checkpoint.pth` |
| `post_train` |  –   |  –   |  –   |  ✓  | `best_model.pth` |
| `full`       |  ✓   |  ✓ (fresh) |  –   |  ✓  | training output |

Autotune overrides below win over the GPU-detected defaults from section 2.

In [ ]:
# ─── Stage gating derived from RUN_MODE (section 1) ─────────────────────────
DO_SMOKE   = RUN_MODE in {'smoke', 'train', 'full'}
DO_TRAIN   = RUN_MODE in {'train', 'resume', 'full'}
DO_RESUME  = RUN_MODE == 'resume'
DO_EVAL    = RUN_MODE in {'post_train', 'full'}
DO_FAILURE = RUN_MODE in {'post_train', 'full'}
DO_TAMPER  = RUN_MODE in {'post_train', 'full'}

# ─── Autotune overrides (None → keep GPU-detected values from § 2) ───────────
# Bump these when § 7's autotune picks too low.
# Heuristic: ResNet34-UNet @ 512² peaks at ~340 MB/sample in the probe;
# real training adds ~500 MB of optimizer + prefetch overhead on top.
#   22 GB (L4):        [24, 32, 48, 56, 64] @ 0.85 → picks 48
#   40 GB (A100):      [16, 24, 32, 48, 64] @ 0.80 → picks 64
#   40 GB, aggressive: [48, 64, 80, 96]     @ 0.86 → picks 80–96
BATCH_CANDIDATES_OVERRIDE = None   # list[int]   e.g. [32, 48, 56, 64]
MEMORY_FRACTION_OVERRIDE  = None   # float 0..1  e.g. 0.88

print(f'RUN_MODE   = {RUN_MODE}')
print(f'DO_SMOKE   = {DO_SMOKE}')
print(f'DO_TRAIN   = {DO_TRAIN}  (resume={DO_RESUME})')
print(f'DO_EVAL    = {DO_EVAL}')
print(f'DO_FAILURE = {DO_FAILURE}')
print(f'DO_TAMPER  = {DO_TAMPER}')
print(f'EPOCHS_OVERRIDE           = {EPOCHS_OVERRIDE}')
print(f'BATCH_CANDIDATES_OVERRIDE = {BATCH_CANDIDATES_OVERRIDE}')
print(f'MEMORY_FRACTION_OVERRIDE  = {MEMORY_FRACTION_OVERRIDE}')

## 4. Clone repo from GitHub & install dependencies

Clones (or pulls) `chautuanminh/RESNET_h95` into `/content/RESNET_h95`, then installs requirements. By default `torch` / `torchvision` are left out of the install so Colab's prebuilt CUDA wheels stay intact (set `REINSTALL_TORCH = True` in section 1 to override). Finally makes the clone importable.

In [ ]:
import os, sys, subprocess

# Clone once, else pull latest.
if not REPO_DIR.exists():
    print(f'Cloning {REPO_URL} → {REPO_DIR} ...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f'Repo present — pulling latest in {REPO_DIR} ...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

assert (REPO_DIR / 'requirements.txt').exists(), f'requirements.txt missing in clone: {REPO_DIR}'
assert TAMPER_META.exists(), f'tamper metadata missing in clone: {TAMPER_META}'

# Install deps. Keep Colab's prebuilt CUDA torch/torchvision unless REINSTALL_TORCH.
if REINSTALL_TORCH:
    subprocess.run(['pip', '-q', 'install', '-r', str(REPO_DIR / 'requirements.txt')], check=True)
else:
    reqs = [ln for ln in (REPO_DIR / 'requirements.txt').read_text().splitlines()
            if ln.strip() and not ln.strip().lower().startswith(('torch', 'torchvision'))]
    Path('/content/reqs_notorch.txt').write_text('\n'.join(reqs) + '\n')
    print('Installing (torch/torchvision left intact):', reqs)
    subprocess.run(['pip', '-q', 'install', '-r', '/content/reqs_notorch.txt'], check=True)

# Make the clone importable.
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f'cwd:         {os.getcwd()}')
print(f'sys.path[0]: {sys.path[0]}')
print('src contents:', sorted(p.name for p in (REPO_DIR / 'src').iterdir()))

## 5. Build the Colab config

Loads the shipped `configs/resnet_h95_config.yaml` from the clone, overrides Drive paths + GPU knobs + (optionally) `training.epochs`, and writes `/content/resnet_h95_colab.yaml`.

Two stage-aware tweaks land here:
- `tampering_type_analysis.enabled` is forced to match `DO_TAMPER` so the in-pipeline validator doesn't expect tamper outputs when the stage is off.
- `validation.require_outputs` flips to `'auto'` whenever any post-train stage is skipped, so `validate_experiment` only enforces the artefacts the enabled stages actually produced.

In [ ]:
from src.config import load_config, deep_set, dump_config

config = load_config('configs/resnet_h95_config.yaml')

config = deep_set(config, 'data.root',                str(DATA_ROOT))
config = deep_set(config, 'data.tamper_metadata_dir', str(TAMPER_META))
config = deep_set(config, 'output.root_dir',          str(OUTPUT_ROOT))
config = deep_set(config, 'output.run_dir',           RUN_DIR)

# Resolve autotune setup: per-session overrides win over GPU-detected defaults.
batch_candidates_final = list(BATCH_CANDIDATES_OVERRIDE) if BATCH_CANDIDATES_OVERRIDE is not None else BATCH_CANDIDATES
memory_fraction_final  = float(MEMORY_FRACTION_OVERRIDE) if MEMORY_FRACTION_OVERRIDE  is not None else MEMORY_FRACTION
if BATCH_CANDIDATES_OVERRIDE is not None:
    print(f'gpu.batch_size_candidates overridden → {batch_candidates_final}')
if MEMORY_FRACTION_OVERRIDE is not None:
    print(f'gpu.auto_tune_memory_fraction overridden → {memory_fraction_final}')

config = deep_set(config, 'gpu.amp',                       True)
config = deep_set(config, 'gpu.amp_dtype',                 AMP_DTYPE)
config = deep_set(config, 'gpu.tf32',                      True)
config = deep_set(config, 'gpu.channels_last',             True)
config = deep_set(config, 'gpu.cudnn_benchmark',           True)
config = deep_set(config, 'gpu.torch_compile',             False)
config = deep_set(config, 'gpu.auto_tune_batch_size',      True)
config = deep_set(config, 'gpu.auto_tune_memory_fraction', memory_fraction_final)
config = deep_set(config, 'gpu.batch_size_candidates',     batch_candidates_final)

config = deep_set(config, 'training.batch_size',                 32)
config = deep_set(config, 'training.num_workers',                4)
config = deep_set(config, 'training.pin_memory',                 True)
config = deep_set(config, 'training.persistent_workers',         True)
config = deep_set(config, 'training.prefetch_factor',            4)
config = deep_set(config, 'training.gradient_accumulation_steps', 1)

if EPOCHS_OVERRIDE is not None:
    config = deep_set(config, 'training.epochs', int(EPOCHS_OVERRIDE))
    print(f'training.epochs overridden → {EPOCHS_OVERRIDE}')

# Stage-aware: tell the validator what to expect.
config = deep_set(config, 'tampering_type_analysis.enabled', bool(DO_TAMPER))
all_post_train_on = DO_EVAL and DO_FAILURE and DO_TAMPER
config = deep_set(config, 'validation.require_outputs', True if all_post_train_on else 'auto')

CONFIG_PATH = '/content/resnet_h95_colab.yaml'
Path(CONFIG_PATH).write_text(dump_config(config), encoding='utf-8')
print(f'Wrote {CONFIG_PATH}')
print()
print(dump_config(config))

## 6. Smoke check — LMDB read + H95 stats + mask coverage  *(DO_SMOKE)*

Walks the train folder and each test folder, reads two samples, prints H95 min/mean/max and mask positive ratio. Fails loudly if LMDB key schemes drift or any path is wrong. Runs for `RUN_MODE` in `smoke` / `train` / `full`.

In [ ]:
if DO_SMOKE:
    from src.smoke import run_smoke
    reports = run_smoke(CONFIG_PATH, sample_count=2)
    print(f'\n✓ Smoke OK: {len(reports)} dataset(s) checked')
else:
    print('• smoke skipped for this RUN_MODE')

## 7. Training  *(DO_TRAIN)*

When enabled, `run_train` performs:
1. `resolve_runtime` — stamps TF32, cuDNN benchmark, channels-last, AMP dtype.
2. `autotune_batch_size` — probes each candidate with a real forward+backward at `[B, 2, 512, 512]`, writes `batch_size_autotune.csv`, picks the largest under `auto_tune_memory_fraction`.
3. `_assert_model_contract` — forwards `[2, 2, 512, 512]`, asserts logits `[2, 1, 512, 512]`.
4. The full loop (epochs from `training.epochs` or `EPOCHS_OVERRIDE`) with AMP + grad clip + cosine LR; updates `train_metrics.csv`, `val_metrics.csv`, `plots/training_curves.png`, `checkpoints/{last,best}_*.pth` every epoch.

- `RUN_MODE='resume'` continues from `<RUN_DIR>/checkpoints/last_checkpoint.pth` on Drive.
- `RUN_MODE='post_train'` skips training and loads `<RUN_DIR>/checkpoints/best_model.pth` for the downstream stages.

In [ ]:
CKPT_DIR = OUTPUT_ROOT / RUN_DIR / 'checkpoints'

resume_ckpt = None
if DO_RESUME:
    resume_ckpt = str(CKPT_DIR / 'last_checkpoint.pth')
    assert Path(resume_ckpt).exists(), (
        f"RUN_MODE='resume' but no checkpoint to resume from: {resume_ckpt}. "
        'Run training first, or switch RUN_MODE to "train".'
    )
    print(f'Resuming from: {resume_ckpt}')

if DO_TRAIN:
    from src.train import run_train
    checkpoint = str(run_train(CONFIG_PATH, resume=resume_ckpt))
    print(f'\n✓ Training complete. Best checkpoint: {checkpoint}')
else:
    # post_train (or smoke) — score an already-trained model.
    checkpoint = str(CKPT_DIR / 'best_model.pth')
    if DO_EVAL or DO_FAILURE or DO_TAMPER:
        assert Path(checkpoint).exists(), (
            f'No trained model for post-train stages: {checkpoint}. '
            'Train first (RUN_MODE="full"/"train") before RUN_MODE="post_train".'
        )
        print(f'• Using existing checkpoint: {checkpoint}')
    else:
        print('• No training and no post-train stages for this RUN_MODE.')

## 8. Evaluation  *(DO_EVAL)*

`src.evaluate.run_evaluate` runs the model on TestingSet, FCD, SCD, writes per-image metrics, threshold sweeps, official 0.5 + best-threshold tables, diagnostic example panels, and bar-chart plots.

In [ ]:
if DO_EVAL:
    from src.evaluate import run_evaluate
    eval_rows = run_evaluate(CONFIG_PATH, checkpoint)
    print(f'\n✓ Evaluation complete. {len(eval_rows)} test-set row(s).')
else:
    print('• eval skipped for this RUN_MODE')

## 9. Failure case analysis  *(DO_FAILURE)*

`src.failure_analysis.run_failure_analysis` ranks the worst-K predictions per test set, writes summaries by category / reason / severity, and saves the worst-200 visualisations.

In [ ]:
if DO_FAILURE:
    from src.failure_analysis import run_failure_analysis
    run_failure_analysis(CONFIG_PATH, checkpoint)
    print('\n✓ Failure analysis complete.')
else:
    print('• failure analysis skipped for this RUN_MODE')

## 10. Tamper-type analysis  *(DO_TAMPER)*

`src.tampering_type_analysis.run_tampering_type_analysis` groups every prediction by the metadata-assigned tamper type (with heuristic fallback) and writes per-type metrics, threshold sweeps, and best-threshold tables.

In [ ]:
if DO_TAMPER:
    from src.tampering_type_analysis import run_tampering_type_analysis
    run_tampering_type_analysis(CONFIG_PATH, checkpoint)
    print('\n✓ Tamper-type analysis complete.')
else:
    print('• tamper-type analysis skipped for this RUN_MODE')

## 11. Summary + validate

Always-on wrap-up: writes the run-level `RES_SUMMARY.md` / `output.md`, then calls `validate_experiment`. With the stage-aware config from § 5, the validator only enforces the artefacts that the stages you ran actually produced.

In [ ]:
from src.output_summary import write_output_summary
from src.validate_experiment import validate_experiment

write_output_summary(CONFIG_PATH)
ok, errors = validate_experiment(CONFIG_PATH)
if ok:
    print('✓ validate_experiment: PASS')
else:
    print('⚠ validate_experiment: FAIL')
    for err in errors:
        print(f'  - {err}')
print(f'\nAll outputs under: {OUTPUT_ROOT / RUN_DIR}')

## 12. Notes

### Where outputs land
Everything is written under `/content/drive/Othercomputers/MSI/results/RESNET_h95/runs/<RUN_DIR>/`. Because this is a Drive-mirrored *"Other computer"* path, files appear on the MSI machine through Drive for Desktop as they're flushed.

Top-level artefacts (relative to `OUTPUT_ROOT/RUN_DIR`):
- `training.log`, `config_snapshot.yaml`, `config_resolved.yaml`, `batch_size_autotune.csv`, `gpu_profile.txt`
- `train_metrics.csv`, `val_metrics.csv`, `plots/training_curves.png`
- `splits/train_indices_seed42.csv`, `splits/val_indices_seed42.csv`
- `checkpoints/best_model.pth`, `checkpoints/last_checkpoint.pth`
- `summary_*.csv`, `official_threshold_0.5_metrics.csv`, `best_threshold_metrics.csv`, `threshold_sweep.csv`, `per_image_metrics.csv`
- `per_image_metrics/`, `threshold_sweeps/`, `examples/`
- `failure_case_analysis/` (only if § 9 ran), `tampering_type_analysis/` (only if § 10 ran)

### Resuming after a disconnect
Checkpoints persist on Drive, so just set `RUN_MODE = 'resume'` in section 1 and run the notebook top to bottom. It continues from `checkpoints/last_checkpoint.pth`.

### Re-running only the post-train stages
Set `RUN_MODE = 'post_train'` in section 1 and run top to bottom — eval / failure / tamper run against `checkpoints/best_model.pth` with no training.

### Encoder weight cache
`TORCH_HOME` points at Drive, so the ImageNet ResNet34 encoder weights download once and are reused on later sessions (no re-download).

### Pushing the batch size higher
If a full epoch finishes with peak VRAM well below the GPU limit (see `gpu_max_reserved_gb` in `train_metrics.csv`), set the autotune overrides in § 3, e.g. `BATCH_CANDIDATES_OVERRIDE = [64, 80, 96]` and `MEMORY_FRACTION_OVERRIDE = 0.86`.

### Code source
Code is canonical on GitHub (`https://github.com/chautuanminh/RESNET_h95`). The clone cell pulls latest each run; there is no Drive code mirror. To test local edits, edit files under `/content/RESNET_h95` and re-run — but commit/push real changes to GitHub.